[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/03_Core_Agent_Anatomy/core_agent_anatomy.ipynb)

# Module 03: Core Agent Anatomy – Deconstructing the Autonomous System

This notebook takes a build-from-scratch approach to agentic systems, using only Vanilla Python and Pydantic. No high-level agent frameworks are used.

## 1. The Body Metaphor: Agent Anatomy

We can understand an autonomous agent by analogy to the human body:

- **Brain (Model):** The reasoning engine (e.g., LLM or logic module).
- **Hands (Tools):** Functions that interact with the world (APIs, calculators, search, etc.).
- **Nervous System (Orchestration):** The control loop that coordinates reasoning and action.
- **Body/Legs (Deployment):** The environment where the agent operates (CLI, web, API, etc.).

## 2. Tools: The 'Hands' Loop

Tools are granular functions that the agent can call to interact with the world. Each tool should have a clear, single responsibility (e.g., `add`, `search_web`, not `manage_finance`).

We implement a tool-calling system from scratch using Pydantic for schema validation.

In [ ]:
from pydantic import BaseModel, Field
from typing import Any, Callable, Dict

class ToolSchema(BaseModel):
    name: str
    description: str
    args_schema: Dict[str, Any]

class Tool:
    def __init__(self, schema: ToolSchema, func: Callable):
        self.schema = schema
        self.func = func

    def invoke(self, **kwargs):
        # Validate arguments using the schema
        for arg, typ in self.schema.args_schema.items():
            if arg not in kwargs:
                raise ValueError(f"Missing argument: {arg}")
            if not isinstance(kwargs[arg], typ):
                raise TypeError(f"Argument '{arg}' must be {typ}")
        return self.func(**kwargs)

# Example: Define a simple calculator tool
calc_schema = ToolSchema(
    name="calculator",
    description="Add two numbers.",
    args_schema={"a": int, "b": int}
)
def add(a: int, b: int) -> int:
    return a + b
calculator_tool = Tool(calc_schema, add)

# Three-Part Loop: Define, Invoke, Observe
result = calculator_tool.invoke(a=2, b=3)
print(f"Calculator result: {result}")

## 3. Orchestration: The 'Nervous System'

The orchestrator is the control loop that coordinates reasoning and action. Here, we implement a simple ReAct (Reason + Act) orchestrator using a Python while loop.